In [1]:
# Cargamos el dataset de flores Iris
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris(as_frame=True)

# Guardamos los datos en un DataFrame
df = iris.frame

# Vemos las primeras filas
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [2]:
# Comprobamos cuántas flores tenemos y si faltan datos
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [3]:
# Vemos los nombres de las especies
print(iris.target_names)

# Contamos cuántas flores hay de cada especie
print(df["target"].value_counts())

['setosa' 'versicolor' 'virginica']
target
0    50
1    50
2    50
Name: count, dtype: int64


In [4]:
# Separamos las medidas de las flores y su especie
X = df.drop(columns=["target"])
y = df["target"]

# Dividimos los datos para entrenamiento y prueba
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Flores para entrenar:", len(X_train))
print("Flores para probar:", len(X_test))

Flores para entrenar: 120
Flores para probar: 30


In [5]:
# Creamos nuestro árbol de decisión
from sklearn.tree import DecisionTreeClassifier

modelo = DecisionTreeClassifier(random_state=42)

# Entrenamos el modelo con las 120 flores
modelo.fit(X_train, y_train)

print("Árbol de decisión entrenado correctamente")

Árbol de decisión entrenado correctamente


In [6]:
# Hacemos predicciones con las flores de prueba
from sklearn.metrics import accuracy_score, classification_report

predicciones = modelo.predict(X_test)

# Comprobamos cuántas flores identifica correctamente
print("Accuracy:", accuracy_score(y_test, predicciones))

# Vemos los resultados por especie
print(classification_report(
    y_test,
    predicciones,
    target_names=iris.target_names
))

Accuracy: 0.9333333333333333
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



In [7]:
# Guardamos nuestro modelo para utilizarlo en Flask
import pickle
from pathlib import Path

ruta = Path("models") if Path("models").is_dir() else Path("../models")

with open(ruta / "modelo_iris.pkl", "wb") as archivo:
    pickle.dump(modelo, archivo)

print("Modelo guardado correctamente")

Modelo guardado correctamente


In [ ]:
# Importamos Flask y las herramientas necesarias
from flask import Flask, request, render_template
import pickle
from pathlib import Path

# Creamos nuestra aplicación
app = Flask(__name__)

# Cargamos el modelo que entrenamos anteriormente
ruta = Path(__file__).resolve().parent.parent / "models" / "modelo_iris.pkl"

with open(ruta, "rb") as archivo:
    modelo = pickle.load(archivo)

# Nombres de las especies
especies = {
    0: "Iris setosa",
    1: "Iris versicolor",
    2: "Iris virginica"
}

@app.route("/", methods=["GET", "POST"])
def index():
    resultado = None

    if request.method == "POST":

        # Recogemos las cuatro medidas de la flor
        medidas = [
            float(request.form["val1"]),
            float(request.form["val2"]),
            float(request.form["val3"]),
            float(request.form["val4"])
        ]

        # Nuestro modelo identifica la especie
        prediccion = modelo.predict([medidas])[0]
        resultado = especies[prediccion]

    return render_template("index.html", prediction=resultado)

NameError: name '__file__' is not defined

Bad pipe message: %s [b'E']
Bad pipe message: %s [b'c\x0b\x9eJ\xbe\x99Q~\xaf0\xf1U\xfa5\xa0\x00\x00\x84\x003\x009\x005\x00/\xc0,\xc00\x00\xa3\x00\x9f\xcc\xa9\xcc\xa8\xcc\xaa\xc0\xaf\xc0\xad\xc0\xa3\xc0\x9f\xc0+\xc0/\x00\xa2\x00\x9e\xc0\xae\xc0\xac\xc0\xa2\xc0\x9e\xc0$\xc0(\x00', b"j\xc0s\xc0w\x00\xc4\x00\xc3\xc0#\xc0'\x00g\x00@\xc0r\xc0v\x00\xbe\x00\xbd\xc0\n\xc0\x14\x008\x00\x88\x00\x87\xc0\t\xc0\x13\x002\x00\x9a\x00\x99\x00E\x00D\x00\x9d\xc0\xa1\xc0\x9d\x00\x9c\xc0\xa0\xc0\x9c\x00=\x00\xc0\x00<\x00\xba\x00\x84\x00\x96\x00A\x00\x07\x00\xff\x01\x00\x00F\x00\x0b\x00\x04\x03\x00\x01\x02\x00\n\x00\n\x00\x08\x00\x1d\x00\x17\x00\x19\x00\x18\x00"]
Bad pipe message: %s [b'\x00\x00\x16\x00\x00\x00\x17\x00\x00\x00\r\x00 \x00\x1e\x06\x01\x06\x02\x06\x03\x05\x01\x05\x02\x05\x03\x04\x01\x04\x02\x04\x03\x03']
Bad pipe message: %s [b'', b'\x03', b'\x02', b'\x03']
Bad pipe message: %s [b'f3\xc8\xaf{\xf5\x86Q\xaa/\n\x88:r9\x1a\x19\xc0\x00\x00\x80\x00\x1d\x00\x1c\xfe\xff\xfe\xfe\x00c\x00e\x00\x11\x00r\